# Model Viewer App - One Click Install

Run all cells to deploy the **Model Viewer** app to your workspace. With no
`session_id`, the notebook launches itself as a Databricks **job** (so the deploy
runs unattended) prints the run URL, then **waits for the job to finish**, pulsing a liveness line every 5s
and pointing at the job URL; when it completes it prints the **app URL**. The job is tagged under
`dbx_vibe_agent_installer_` (`source`, `app`, `duration`, `session_id`). It creates the app, deploys it, and prints the shareable URL.

In [ ]:
# === JobLauncher: launch this notebook as a tagged Databricks job (agent pattern) ===
import re as _jl_re


class JobLauncher:
    """Create/reuse a named Databricks job for this notebook and run it, with tags.

    Mirrors the agent's launcher: build with the notebook path, the widget values to
    pass as base_parameters, and a {tag_key: tag_value} dict attached to the job.
    """

    _TAG_SAFE_RE = _jl_re.compile(r"[^A-Za-z0-9._-]")

    @staticmethod
    def _sanitize_tag(value):
        s = JobLauncher._TAG_SAFE_RE.sub("_", str(value))
        s = _jl_re.sub(r"_+", "_", s)
        return s.strip("_").strip(".").strip("-")

    def __init__(self, notebook_path, widget_key_values, job_tags=None):
        self.notebook_path = str(notebook_path)
        self.widget_key_values = {str(k): str(v) for k, v in widget_key_values.items()}
        raw = dict(job_tags or {})
        self.job_tags = {self._sanitize_tag(k): self._sanitize_tag(v) for k, v in raw.items()}

    @staticmethod
    def _detect_compute_type():
        """Detect serverless vs classic (ported from the agent's proven launcher).

        Serverless compute exposes a non-empty `clusterUsageTags.clusterId`, so a
        clusterId-only check misfires and wrongly attaches a classic cluster - which a
        serverless-only workspace rejects ("Only serverless compute is supported").
        The reliable signals are the IS_SERVERLESS env var and the absence of a
        cluster *name*. Returns (is_serverless, cluster_id)."""
        import os as _jl_os
        if _jl_os.environ.get("IS_SERVERLESS", "").upper() == "TRUE":
            return True, None
        try:
            spark.conf.get("spark.databricks.clusterUsageTags.clusterName")
        except Exception:
            return True, None
        try:
            _cid = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
            if _cid:
                return False, _cid
        except Exception:
            pass
        return True, None

    @staticmethod
    def _get_workspace_context():
        host, org = "", ""
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            try:
                host = ctx.apiUrl().get()
            except Exception:
                pass
            try:
                org = ctx.workspaceId().get()
            except Exception:
                pass
        except Exception:
            pass
        if not host:
            try:
                from databricks.sdk import WorkspaceClient as _WC
                host = str(_WC().config.host)
            except Exception:
                pass
        return (host or "").rstrip("/"), org

    @staticmethod
    def get_current_notebook_path():
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            try:
                p = ctx.notebookPath().get()
                if p:
                    return p
            except Exception:
                pass
            import json as _j
            c = _j.loads(ctx.toJson())
            for k in ("notebook_path", "notebookPath"):
                v = (c.get("extraContext") or {}).get(k, "") or (c.get("tags") or {}).get(k, "")
                if v:
                    return v
        except Exception:
            pass
        return ""

    def launch(self, job_name=None, run_name=None):
        import time as _t
        from databricks.sdk import WorkspaceClient as _WC
        from databricks.sdk.service import jobs as _jobs
        out = {"success": False, "job_id": None, "run_id": None, "job_url": "", "error": None}
        try:
            w = _WC()
            job_name = job_name or ("dbx_vibe_installer_%d" % int(_t.time()))
            is_serverless, cluster_id = self._detect_compute_type()

            def _build_task(attach_cluster):
                t = _jobs.Task(
                    task_key="install",
                    notebook_task=_jobs.NotebookTask(
                        notebook_path=self.notebook_path,
                        base_parameters=self.widget_key_values),
                    timeout_seconds=14400,
                )
                if attach_cluster and cluster_id:
                    t.existing_cluster_id = cluster_id
                return t

            existing = None
            try:
                for j in w.jobs.list(name=job_name):
                    if j.settings and j.settings.name == job_name:
                        existing = j.job_id
                        break
            except Exception:
                pass

            def _create_and_run(attach_cluster):
                task = _build_task(attach_cluster)
                if existing:
                    w.jobs.reset(job_id=existing,
                                 new_settings=_jobs.JobSettings(name=job_name, tags=self.job_tags, tasks=[task]))
                    jid = existing
                else:
                    jid = w.jobs.create(name=job_name, tags=self.job_tags, tasks=[task]).job_id
                r = w.jobs.run_now(job_id=jid)
                return jid, r

            try:
                job_id, run = _create_and_run(attach_cluster=(not is_serverless))
            except Exception as ce:
                # Defense in depth: if a serverless-only workspace rejects the attached
                # cluster (detection misfired), retry as a pure serverless task.
                if "serverless" in str(ce).lower() and not is_serverless:
                    job_id, run = _create_and_run(attach_cluster=False)
                else:
                    raise
            host, org = self._get_workspace_context()
            url = "%s/jobs/%s/runs/%s%s" % (host, job_id, run.run_id, ("?o=%s" % org if org else "")) if host else ""
            out.update({"success": True, "job_id": job_id, "run_id": run.run_id, "job_url": url})
        except Exception as e:
            out["error"] = str(e)
        return out

    @staticmethod
    def wait_for_run(run_id, job_url="", pulse_seconds=5, logger=None):
        """Block until the launched run reaches a terminal state, emitting a liveness
        pulse every pulse_seconds and pointing at the job URL. Returns
        {life_cycle_state, result_state, notebook_output, error}."""
        import time as _t
        from databricks.sdk import WorkspaceClient as _WC
        log = logger or (lambda m: print(m, flush=True))
        w = _WC()
        terminal = {"TERMINATED", "INTERNAL_ERROR", "SKIPPED"}
        out = {"life_cycle_state": "", "result_state": "", "notebook_output": "", "error": None}
        start = _t.time()
        while True:
            try:
                run = w.jobs.get_run(run_id=run_id)
            except Exception as e:
                log("   poll error (%s) - retrying in %ds" % (str(e)[:120], pulse_seconds))
                _t.sleep(pulse_seconds)
                continue
            state = run.state if run else None
            st = state.life_cycle_state.value if (state and state.life_cycle_state) else ""
            rs = state.result_state.value if (state and state.result_state) else ""
            elapsed = int(_t.time() - start)
            if st in terminal:
                out["life_cycle_state"], out["result_state"] = st, rs
                try:
                    trid = run.tasks[0].run_id if (run and run.tasks) else run_id
                    ro = w.jobs.get_run_output(run_id=trid)
                    out["notebook_output"] = (ro.notebook_output.result if ro.notebook_output else "") or ""
                    if ro.error:
                        out["error"] = ro.error
                except Exception as e:
                    out["error"] = str(e)
                return out
            mon = (" | monitor: %s" % job_url) if job_url else ""
            log("   ... job still running [%s] %ds elapsed%s" % (st or "PENDING", elapsed, mon))
            _t.sleep(pulse_seconds)

    @staticmethod
    def update_job_tags(updated_tags):
        """Merge tags onto the job currently running this notebook (best effort)."""
        res = {"success": False, "error": None}
        if not updated_tags:
            res["error"] = "empty"
            return res
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            job_id_str = ""
            try:
                job_id_str = ctx.jobId().get()
            except Exception:
                pass
            if not job_id_str:
                res["error"] = "not running as a job (no jobId)"
                return res
            from databricks.sdk import WorkspaceClient as _WC
            from databricks.sdk.service import jobs as _jobs
            w = _WC()
            job_id = int(job_id_str)
            info = w.jobs.get(job_id=job_id)
            existing = dict(info.settings.tags or {})
            new = {JobLauncher._sanitize_tag(k): JobLauncher._sanitize_tag(v) for k, v in updated_tags.items()}
            merged = {**existing, **new}
            w.jobs.update(job_id=job_id, new_settings=_jobs.JobSettings(tags=merged))
            res["success"] = True
        except Exception as e:
            res["error"] = str(e)
        return res


# === Launcher helpers (same tag family / pattern as the data-model installer) ===
INSTALLER_TAG_PREFIX = "dbx_vibe_agent_installer_"


def _pget(name, default=""):
    """Read a widget/job-parameter value if present, else the default. session_id is
    passed to the launched job as a base parameter (not a visible widget), so it is
    read through this safe getter rather than dbutils.widgets.get, which raises when no
    widget was ever created in an interactive run."""
    try:
        v = dbutils.widgets.get(name)
    except Exception:
        return default
    return v if (v is not None and v != "") else default


def _gen_session_id():
    import uuid
    return str((uuid.uuid4().int >> 64) & 9223372036854775807)


def _running_as_job():
    """True when this notebook is executing inside a Databricks job run (jobId set).
    Hard backstop so a launched job never re-launches itself, even if its session_id
    job parameter is somehow unreadable."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return bool(ctx.jobId().get())
    except Exception:
        return False


In [ ]:
APP_NAME = "model-viewer-app"
# GitHub fallback: where to fetch the app source if the workspace folder is missing.
# Change these two if you fork this installer for a different repo.
FALLBACK_REPO = "databricks-industry-solutions/lakehouse-industry-data-models"
FALLBACK_BRANCH = "main"
# Candidate app-source folders, tried in order. The first one that exists in the repo
# wins, so this works both after the PR that renames viewer/ -> model-viewer/ is merged
# (model-viewer/...) and before it is merged (the original viewer/...).
FALLBACK_SUBPATHS = ["model-viewer/model-viewer-app", "viewer/model-viewer-app"]

import time, requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host, token = ctx.apiUrl().get(), ctx.apiToken().get()
H = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

nb = ctx.notebookPath().get()
ws_logical = "/".join(nb.split("/")[:-1]) + "/" + APP_NAME
src_fs = ws_logical if ws_logical.startswith("/Workspace/") else "/Workspace" + ws_logical


def _local_app_present():
    r = requests.get(f"{host}/api/2.0/workspace/get-status", headers=H,
                     params={"path": f"{ws_logical}/app.yaml"})
    return r.status_code == 200, r


def _install_from_github():
    """Fetch the first existing FALLBACK_SUBPATHS candidate from FALLBACK_REPO@FALLBACK_BRANCH and import every
    file into ws_logical. Raises Exception with a clear cause on any failure
    (network unreachable, repo/branch 404, no files matched, upload error)."""
    gh_h = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
    tree_url = (f"https://api.github.com/repos/{FALLBACK_REPO}/git/trees/"
                f"{FALLBACK_BRANCH}?recursive=1")
    try:
        tr = requests.get(tree_url, headers=gh_h, timeout=15)
    except requests.exceptions.RequestException as ne:
        raise Exception(f"network unreachable while fetching {tree_url}: {ne}")
    if tr.status_code == 404:
        raise Exception(f"repo or branch not found: {FALLBACK_REPO}@{FALLBACK_BRANCH} "
                        f"(GitHub returned 404 for {tree_url})")
    if tr.status_code in (401, 403):
        raise Exception(f"GitHub denied access to {FALLBACK_REPO}@{FALLBACK_BRANCH} "
                        f"({tr.status_code}); is the repo private or rate-limited? "
                        f"Body: {tr.text[:200]}")
    if tr.status_code != 200:
        raise Exception(f"unexpected GitHub response {tr.status_code} for {tree_url}: "
                        f"{tr.text[:200]}")

    tree = tr.json().get("tree") or []
    # Pick the first candidate subpath that actually contains files, so the installer
    # works whether the repo uses the new 'model-viewer/' folder or the old 'viewer/'.
    prefix, chosen = None, None
    for cand in FALLBACK_SUBPATHS:
        cp = cand.rstrip("/") + "/"
        if any(e.get("type") == "blob" and (e.get("path") or "").startswith(cp) for e in tree):
            prefix, chosen = cp, cand
            break
    if prefix is None:
        raise Exception(f"no files found under any of {FALLBACK_SUBPATHS} in "
                        f"{FALLBACK_REPO}@{FALLBACK_BRANCH} (tree returned {len(tree)} entries total)")
    files = [e for e in tree
             if e.get("type") == "blob" and (e.get("path") or "").startswith(prefix)]

    print(f"       Fetching {len(files)} file(s) from {FALLBACK_REPO}@{FALLBACK_BRANCH}/{chosen}/ ...")
    mr = requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=H, json={"path": ws_logical})
    if mr.status_code not in (200, 201):
        raise Exception(f"workspace mkdirs failed for {ws_logical}: {mr.status_code} {mr.text[:200]}")

    for entry in files:
        rel = entry["path"][len(prefix):]
        if not rel:
            continue
        try:
            br = requests.get(
                f"https://api.github.com/repos/{FALLBACK_REPO}/git/blobs/{entry['sha']}",
                headers=gh_h, timeout=30,
            )
        except requests.exceptions.RequestException as ne:
            raise Exception(f"network unreachable while fetching blob {rel}: {ne}")
        if br.status_code != 200:
            raise Exception(f"blob fetch failed for {rel} ({br.status_code}): {br.text[:200]}")
        content_b64 = (br.json().get("content") or "").replace("\n", "")
        if not content_b64:
            raise Exception(f"blob {rel} returned empty content (size={entry.get('size')})")

        target = f"{ws_logical}/{rel}"
        parent = target.rsplit("/", 1)[0]
        if parent != ws_logical:
            requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=H, json={"path": parent})
        ir = requests.post(
            f"{host}/api/2.0/workspace/import", headers=H,
            json={"path": target, "format": "AUTO", "content": content_b64, "overwrite": True},
        )
        if ir.status_code not in (200, 201):
            raise Exception(f"workspace import failed for {target}: {ir.status_code} {ir.text[:200]}")
        print(f"         + {rel}")

    ok, _ = _local_app_present()
    if not ok:
        raise Exception(f"files uploaded but app.yaml still not visible at {ws_logical}/app.yaml")


def _wait_compute_active(timeout_s=600):
    deadline = time.time() + timeout_s
    last = ""
    while time.time() < deadline:
        info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
        cs = info.get("compute_status", {}).get("state", "")
        if cs != last:
            print(f"       Compute: {cs}")
            last = cs
        if cs == "ACTIVE":
            return info
        if cs in ("ERROR", "STOPPED"):
            raise Exception(f"Compute entered {cs} while waiting; aborting deploy.")
        time.sleep(10)
    raise Exception(f"Compute did not reach ACTIVE within {timeout_s}s (last state={last!r}).")


def _ensure_source():
    local_ok, local_resp = _local_app_present()
    if local_ok:
        print(f"[1/4] Source verified locally: {ws_logical}")
    else:
        print(f"[1/4] Local source not found at {ws_logical}/app.yaml "
              f"(workspace get-status: {local_resp.status_code}); "
              f"falling back to GitHub: {FALLBACK_REPO}@{FALLBACK_BRANCH} {FALLBACK_SUBPATHS}")
        try:
            _install_from_github()
        except Exception as fe:
            raise Exception(
                f"App source not available.\n"
                f"  - Local check FAILED: {ws_logical}/app.yaml not found in workspace.\n"
                f"  - GitHub fallback FAILED: {fe}\n"
                f"Fix one of the following:\n"
                f"  (a) Sync the 'model-viewer/' (or 'viewer/') folder from the repo into your workspace alongside this notebook, OR\n"
                f"  (b) Ensure this cluster can reach api.github.com and the repo "
                f"'{FALLBACK_REPO}@{FALLBACK_BRANCH}' is accessible.\n"
            )
        print(f"       ✓ App source materialized at {ws_logical}")
    print(f"       Apps API source_code_path: {src_fs}")


def deploy():
    """Ensure the app source is present, create/start the app, deploy a
    SNAPSHOT, wait for SUCCEEDED, and return the live app URL."""
    _ensure_source()

    r = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H)
    if r.status_code == 200:
        print(f"[2/4] App '{APP_NAME}' exists, ensuring compute is running...")
        cs = r.json().get("compute_status", {}).get("state", "")
        if cs != "ACTIVE":
            if cs in ("STOPPED", "ERROR", ""):
                sr = requests.post(f"{host}/api/2.0/apps/{APP_NAME}/start", headers=H, json={})
                assert sr.status_code in (200, 201, 202), f"Failed to start app compute: {sr.status_code} {sr.text}"
            _wait_compute_active()
    elif r.status_code == 404:
        print(f"[2/4] Creating app '{APP_NAME}'...")
        cr = requests.post(f"{host}/api/2.0/apps", headers=H, json={"name": APP_NAME, "description": "Data Model Viewer - interactive graph visualization"})
        assert cr.status_code in (200, 201), f"Failed to create app: {cr.status_code} {cr.text}"
        print(f"       App created. Waiting for compute to become ACTIVE...")
        _wait_compute_active()
    else:
        raise Exception(f"Unexpected response checking app: {r.status_code} {r.text}")

    print(f"[3/4] Deploying...")
    r = requests.post(
        f"{host}/api/2.0/apps/{APP_NAME}/deployments",
        headers=H,
        json={"source_code_path": src_fs, "mode": "SNAPSHOT"},
    )
    assert r.status_code in (200, 201), f"Deploy failed: {r.status_code} {r.text}"
    deployment_id = r.json().get("deployment_id", "")
    print(f"       Deployment started: {deployment_id}")

    last_state = ""
    for _ in range(60):
        time.sleep(10)
        info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
        dep = info.get("active_deployment") or info.get("pending_deployment") or {}
        status = dep.get("status", {})
        state = status.get("state", "")
        msg = status.get("message", "")
        if state != last_state:
            print(f"       {state}: {msg}")
            last_state = state
        if state == "SUCCEEDED":
            url = info.get("url", "")
            app_state = info.get("app_status", {}).get("state", "")
            print(f"[4/4] Deployed (app_status={app_state}).")
            print(f"\n{'='*60}\n  App URL: {url}\n{'='*60}")
            try:
                displayHTML(
                    f'<h2 style="color:#4ECDC4">Model Viewer Deployed!</h2>'
                    f'<p style="font-size:16px"><a href="{url}" target="_blank">{url}</a></p>'
                    f'<p>Upload a <code>model.json</code> to visualize your data model.</p>'
                )
            except Exception:
                pass
            return url
        if state == "FAILED":
            raise Exception(f"Deployment failed: {msg}")
    else:
        raise Exception("Deployment did not complete within 10 minutes. Check app status in the workspace UI.")


# === Launcher gate + tagged run ===
def launch_and_wait():
    """Launch this notebook as a tagged Databricks job, then BLOCK until the deploy
    job finishes, emitting a liveness pulse every 5s and pointing at the job URL.
    On completion, surface the app URL. Returns (handled, success, exit_value)."""
    sid = _gen_session_id()
    p = INSTALLER_TAG_PREFIX
    tags = {
        p + "source": "Model_Viewer_App_Installer",
        p + "app": APP_NAME,
        p + "duration": "running",
        p + "session_id": sid,
    }
    nb_path = JobLauncher.get_current_notebook_path()
    if not nb_path:
        print("Could not determine notebook path; deploying directly instead.")
        return (False, False, None)
    job_name = "dbx_vibe_installer_viewer_app"
    print("No session_id -> launching the app install as a Databricks job (%s) ..." % job_name)
    res = JobLauncher(nb_path, {"session_id": sid}, tags).launch(job_name=job_name, run_name=job_name)
    if not res["success"]:
        print("Job launch failed (%s) -> deploying directly instead." % res.get("error"))
        return (False, False, None)

    job_url = res["job_url"] or ("job_id=%s run_id=%s" % (res["job_id"], res["run_id"]))
    _plog = lambda m: print(m, flush=True)
    print("=" * 60)
    print("A Databricks JOB is deploying the Model Viewer app (%s)." % APP_NAME)
    print("Monitor it live here: %s" % job_url)
    print("Waiting for the job to finish (liveness pulse every 5s) ...")
    print("=" * 60)

    r = JobLauncher.wait_for_run(res["run_id"], job_url=job_url, pulse_seconds=5, logger=_plog)
    app_url = ""
    try:
        app_url = requests.get("%s/api/2.0/apps/%s" % (host, APP_NAME), headers=H).json().get("url", "") or ""
    except Exception:
        app_url = ""
    nbout = r.get("notebook_output") or ""
    print("=" * 60)
    if r.get("result_state") == "SUCCESS":
        print("APP INSTALL JOB COMPLETE (%s)." % r.get("result_state"))
        if nbout:
            print("  Job result: %s" % nbout)
        if app_url:
            print("  Open the app here: %s" % app_url)
        print("=" * 60)
        return (True, True, "SUCCESS via job: %s deployed -> %s | job: %s" % (APP_NAME, app_url or "(n/a)", job_url))
    print("APP INSTALL JOB DID NOT SUCCEED (state=%s, result=%s)." % (r.get("life_cycle_state"), r.get("result_state")))
    if nbout:
        print("  Job result: %s" % nbout)
    if r.get("error"):
        print("  Job error: %s" % str(r.get("error"))[:300])
    print("  Inspect the run here: %s" % job_url)
    print("=" * 60)
    return (True, False, "FAILED via job: %s (result=%s) | job: %s" % (APP_NAME, r.get("result_state"), job_url))


def main():
    # Launcher gate: an interactive run with no session_id launches the deploy as a
    # tagged Databricks job. _running_as_job() is the backstop so a launched job runs
    # the deploy in-place instead of re-launching itself.
    if not _pget("session_id", "").strip() and not _running_as_job():
        handled, ok, exitval = launch_and_wait()
        if handled:
            dbutils.notebook.exit(exitval)
            return

    import time as _t
    t0 = _t.time()
    p = INSTALLER_TAG_PREFIX
    result = None
    try:
        url = deploy()
        elapsed = _t.time() - t0
        tr = JobLauncher.update_job_tags({
            p + "source": "Model_Viewer_App_Installer",
            p + "app": APP_NAME,
            p + "duration": "%.1fmin" % (elapsed / 60.0),
        })
        print("Tag update: %s" % ("ok" if tr["success"] else tr.get("error")))
        result = "SUCCESS: %s deployed -> %s (%.1f min)" % (APP_NAME, url, elapsed / 60.0)
        print(result)
    except Exception:
        try:
            JobLauncher.update_job_tags({
                p + "source": "Model_Viewer_App_Installer",
                p + "app": APP_NAME,
                p + "duration": "%.1fmin" % ((_t.time() - t0) / 60.0),
            })
        except Exception:
            pass
        raise
    dbutils.notebook.exit(result)


main()
